# AI-Powered Malware Analysis

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import hashlib
import struct
import re
import numpy as np
from collections import defaultdict, Counter
import base64
import zlib
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import json

@dataclass
class MalwareSignature:
    name: str
    pattern: bytes
    severity: str
    family: str
    technique: str

class StaticAnalyzer:
    def __init__(self):
        self.pe_signatures = [
            MalwareSignature("CreateRemoteThread", b"CreateRemoteThread", "HIGH", "Injection", "Process Injection"),
            MalwareSignature("VirtualAllocEx", b"VirtualAllocEx", "MEDIUM", "Memory", "Memory Allocation"),
            MalwareSignature("WriteProcessMemory", b"WriteProcessMemory", "HIGH", "Injection", "Memory Writing"),
            MalwareSignature("RegOpenKeyEx", b"RegOpenKeyEx", "MEDIUM", "Persistence", "Registry Access"),
            MalwareSignature("CreateFile", b"CreateFile", "LOW", "FileSystem", "File Operations"),
            MalwareSignature("InternetOpen", b"InternetOpen", "MEDIUM", "Network", "HTTP Communication"),
            MalwareSignature("CryptEncrypt", b"CryptEncrypt", "HIGH", "Ransomware", "File Encryption"),
            MalwareSignature("SetWindowsHookEx", b"SetWindowsHookEx", "HIGH", "Keylogger", "Hook Installation")
        ]

        self.yara_patterns = {
            'packer_upx': rb'UPX[!\x00]',
            'vm_detection': rb'VBoxService|VMwareService|qemu',
            'anti_debug': rb'IsDebuggerPresent|CheckRemoteDebuggerPresent',
            'crypto_constants': rb'\x67\x45\x23\x01|\x01\x23\x45\x67',
            'suspicious_strings': rb'cmd\.exe|powershell\.exe|rundll32\.exe'
        }

    def calculate_entropy(self, data: bytes) -> float:
        if not data:
            return 0.0
        byte_counts = Counter(data)
        entropy = 0.0
        data_len = len(data)
        for count in byte_counts.values():
            p = count / data_len
            entropy -= p * np.log2(p)
        return entropy

    def detect_packers(self, binary_data: bytes) -> List[str]:
        packers = []
        if b'UPX!' in binary_data:
            packers.append('UPX')
        if b'FSG!' in binary_data:
            packers.append('FSG')
        if b'MPRESS' in binary_data:
            packers.append('MPRESS')
        if b'Themida' in binary_data:
            packers.append('Themida')
        return packers

    def extract_strings(self, binary_data: bytes, min_length: int = 4) -> List[str]:
        ascii_pattern = rb'[ -~]{%d,}' % min_length
        unicode_pattern = rb'(?:[ -~]\x00){%d,}' % min_length

        strings = []
        for match in re.finditer(ascii_pattern, binary_data):
            strings.append(match.group(0).decode('ascii', errors='ignore'))

        for match in re.finditer(unicode_pattern, binary_data):
            unicode_str = match.group(0).decode('utf-16le', errors='ignore')
            strings.append(unicode_str)

        return strings[:100]

    def analyze_pe_header(self, binary_data: bytes) -> Dict:
        if len(binary_data) < 64 or binary_data[:2] != b'MZ':
            return {'valid_pe': False}

        try:
            e_lfanew = struct.unpack('<I', binary_data[60:64])[0]
            if e_lfanew + 4 >= len(binary_data):
                return {'valid_pe': False}

            pe_signature = binary_data[e_lfanew:e_lfanew+4]
            if pe_signature != b'PE\x00\x00':
                return {'valid_pe': False}

            machine = struct.unpack('<H', binary_data[e_lfanew+4:e_lfanew+6])[0]
            num_sections = struct.unpack('<H', binary_data[e_lfanew+6:e_lfanew+8])[0]
            timestamp = struct.unpack('<I', binary_data[e_lfanew+8:e_lfanew+12])[0]

            return {
                'valid_pe': True,
                'machine_type': machine,
                'sections': num_sections,
                'timestamp': timestamp,
                'compile_time': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(timestamp))
            }
        except:
            return {'valid_pe': False}

    def signature_scan(self, binary_data: bytes) -> List[Dict]:
        detections = []
        for sig in self.pe_signatures:
            if sig.pattern in binary_data:
                detections.append({
                    'name': sig.name,
                    'severity': sig.severity,
                    'family': sig.family,
                    'technique': sig.technique
                })

        for pattern_name, pattern in self.yara_patterns.items():
            if re.search(pattern, binary_data, re.IGNORECASE):
                detections.append({
                    'name': pattern_name,
                    'severity': 'MEDIUM',
                    'family': 'Generic',
                    'technique': 'Pattern Match'
                })

        return detections

class DynamicAnalyzer:
    def __init__(self):
        self.api_hooks = defaultdict(int)
        self.file_operations = []
        self.network_connections = []
        self.registry_operations = []
        self.process_creations = []

    def simulate_execution(self, binary_data: bytes) -> Dict:
        behavioral_indicators = {}

        if b'CreateProcess' in binary_data:
            self.process_creations.append('cmd.exe')
            behavioral_indicators['process_injection'] = 0.8

        if b'RegSetValue' in binary_data:
            self.registry_operations.append('HKEY_CURRENT_USER\\Software\\Microsoft\\Windows\\CurrentVersion\\Run')
            behavioral_indicators['persistence'] = 0.9

        if b'socket' in binary_data or b'InternetConnect' in binary_data:
            self.network_connections.append('192.168.1.100:4444')
            behavioral_indicators['c2_communication'] = 0.7

        if b'CopyFile' in binary_data or b'MoveFile' in binary_data:
            self.file_operations.append('C:\\Users\\victim\\Documents\\important.docx')
            behavioral_indicators['file_manipulation'] = 0.6

        return behavioral_indicators

    def analyze_network_behavior(self, pcap_data: bytes = None) -> Dict:
        network_analysis = {
            'dns_requests': ['malware-c2.evil.com', 'update.suspicious-domain.net'],
            'http_requests': ['POST /gate.php', 'GET /config.bin'],
            'suspicious_ports': [4444, 8080, 1337],
            'encrypted_traffic': True
        }
        return network_analysis

class MachineLearningClassifier:
    def __init__(self):
        self.features = [
            'file_size', 'entropy', 'num_sections', 'num_imports',
            'num_exports', 'has_debug_info', 'is_packed', 'num_strings'
        ]
        self.weights = np.random.random(len(self.features))
        self.bias = np.random.random()

    def extract_features(self, binary_data: bytes, static_analysis: Dict) -> np.array:
        features = np.zeros(len(self.features))

        features[0] = len(binary_data) / 1000000.0  # Normalize file size
        features[1] = StaticAnalyzer().calculate_entropy(binary_data)
        features[2] = static_analysis.get('sections', 0) / 10.0
        features[3] = len([sig for sig in static_analysis.get('detections', []) if 'import' in sig['name'].lower()])
        features[4] = 0.1  # Placeholder for exports
        features[5] = 1.0 if static_analysis.get('timestamp', 0) > 0 else 0.0
        features[6] = 1.0 if static_analysis.get('packed', False) else 0.0
        features[7] = len(static_analysis.get('strings', [])) / 100.0

        return features

    def predict_malware_family(self, features: np.array) -> Tuple[str, float]:
        families = ['Trojan', 'Ransomware', 'Backdoor', 'Worm', 'Spyware', 'Adware', 'Rootkit']

        scores = np.dot(features, self.weights) + self.bias
        family_scores = np.abs(np.sin(scores * np.arange(len(families))))

        predicted_family = families[np.argmax(family_scores)]
        confidence = np.max(family_scores)

        return predicted_family, min(confidence, 1.0)

    def calculate_threat_score(self, static_results: Dict, dynamic_results: Dict) -> float:
        base_score = 0.0

        if static_results.get('detections'):
            severity_weights = {'HIGH': 0.3, 'MEDIUM': 0.2, 'LOW': 0.1}
            for detection in static_results['detections']:
                base_score += severity_weights.get(detection['severity'], 0.1)

        if dynamic_results:
            base_score += sum(dynamic_results.values()) * 0.4

        entropy = static_results.get('entropy', 0)
        if entropy > 7.5:
            base_score += 0.2

        if static_results.get('packed'):
            base_score += 0.3

        return min(base_score, 1.0)

class MalwareAnalysisEngine:
    def __init__(self):
        self.static_analyzer = StaticAnalyzer()
        self.dynamic_analyzer = DynamicAnalyzer()
        self.ml_classifier = MachineLearningClassifier()
        self.analysis_cache = {}

    def generate_file_hash(self, binary_data: bytes) -> Dict[str, str]:
        return {
            'md5': hashlib.md5(binary_data).hexdigest(),
            'sha1': hashlib.sha1(binary_data).hexdigest(),
            'sha256': hashlib.sha256(binary_data).hexdigest()
        }

    def comprehensive_analysis(self, binary_data: bytes, filename: str = "sample.exe") -> Dict:
        file_hashes = self.generate_file_hash(binary_data)

        if file_hashes['sha256'] in self.analysis_cache:
            return self.analysis_cache[file_hashes['sha256']]

        static_results = {
            'filename': filename,
            'file_size': len(binary_data),
            'hashes': file_hashes,
            'entropy': self.static_analyzer.calculate_entropy(binary_data),
            'pe_analysis': self.static_analyzer.analyze_pe_header(binary_data),
            'strings': self.static_analyzer.extract_strings(binary_data),
            'packers': self.static_analyzer.detect_packers(binary_data),
            'detections': self.static_analyzer.signature_scan(binary_data)
        }

        static_results['packed'] = len(static_results['packers']) > 0

        dynamic_results = self.dynamic_analyzer.simulate_execution(binary_data)
        network_behavior = self.dynamic_analyzer.analyze_network_behavior()

        features = self.ml_classifier.extract_features(binary_data, static_results)
        predicted_family, confidence = self.ml_classifier.predict_malware_family(features)
        threat_score = self.ml_classifier.calculate_threat_score(static_results, dynamic_results)

        final_report = {
            'analysis_timestamp': time.time(),
            'static_analysis': static_results,
            'dynamic_analysis': {
                'behavioral_indicators': dynamic_results,
                'network_behavior': network_behavior
            },
            'ml_classification': {
                'predicted_family': predicted_family,
                'confidence': confidence,
                'threat_score': threat_score
            },
            'risk_assessment': self.generate_risk_assessment(threat_score, static_results, dynamic_results)
        }

        self.analysis_cache[file_hashes['sha256']] = final_report
        return final_report

    def generate_risk_assessment(self, threat_score: float, static: Dict, dynamic: Dict) -> Dict:
        risk_level = "LOW"
        if threat_score > 0.7:
            risk_level = "CRITICAL"
        elif threat_score > 0.5:
            risk_level = "HIGH"
        elif threat_score > 0.3:
            risk_level = "MEDIUM"

        recommendations = []
        if static.get('packed'):
            recommendations.append("Sample is packed - requires unpacking for deeper analysis")
        if len(static.get('detections', [])) > 5:
            recommendations.append("Multiple suspicious API calls detected")
        if dynamic.get('c2_communication', 0) > 0.5:
            recommendations.append("Potential C2 communication detected - isolate network")

        return {
            'risk_level': risk_level,
            'threat_score': threat_score,
            'recommendations': recommendations,
            'mitre_tactics': self.map_mitre_tactics(static, dynamic)
        }

    def map_mitre_tactics(self, static: Dict, dynamic: Dict) -> List[str]:
        tactics = []

        if any('persistence' in d.get('technique', '').lower() for d in static.get('detections', [])):
            tactics.append('T1547 - Boot or Logon Autostart Execution')

        if dynamic.get('process_injection', 0) > 0.5:
            tactics.append('T1055 - Process Injection')

        if dynamic.get('c2_communication', 0) > 0.5:
            tactics.append('T1071 - Application Layer Protocol')

        if any('encrypt' in d.get('name', '').lower() for d in static.get('detections', [])):
            tactics.append('T1486 - Data Encrypted for Impact')

        return tactics

def simulate_malware_analysis():
    analyzer = MalwareAnalysisEngine()

    fake_malware = b'MZ\x90\x00' + b'PE\x00\x00' * 2 + b'CreateRemoteThread' * 3
    fake_malware += b'VirtualAllocEx' + b'RegSetValue' + b'InternetConnect'
    fake_malware += b'CryptEncrypt' * 2 + b'UPX!' + b'\x90' * 1000
    fake_malware += b'malware-c2.evil.com' + b'cmd.exe' + b'\x00' * 500

    print("=== Advanced Malware Analysis Framework ===\n")

    report = analyzer.comprehensive_analysis(fake_malware, "suspicious_sample.exe")

    print(f"File: {report['static_analysis']['filename']}")
    print(f"SHA256: {report['static_analysis']['hashes']['sha256'][:16]}...")
    print(f"Size: {report['static_analysis']['file_size']} bytes")
    print(f"Entropy: {report['static_analysis']['entropy']:.3f}")

    print(f"\n--- ML Classification ---")
    ml_results = report['ml_classification']
    print(f"Predicted Family: {ml_results['predicted_family']}")
    print(f"Confidence: {ml_results['confidence']:.3f}")
    print(f"Threat Score: {ml_results['threat_score']:.3f}")

    print(f"\n--- Static Analysis ---")
    detections = report['static_analysis']['detections']
    print(f"Signatures Detected: {len(detections)}")
    for detection in detections[:5]:
        print(f"  {detection['name']} [{detection['severity']}] - {detection['technique']}")

    if report['static_analysis']['packers']:
        print(f"Packers: {', '.join(report['static_analysis']['packers'])}")

    print(f"\n--- Dynamic Analysis ---")
    behavioral = report['dynamic_analysis']['behavioral_indicators']
    for behavior, score in behavioral.items():
        print(f"  {behavior.replace('_', ' ').title()}: {score:.3f}")

    print(f"\n--- Risk Assessment ---")
    risk = report['risk_assessment']
    print(f"Risk Level: {risk['risk_level']}")
    print(f"MITRE Tactics: {len(risk['mitre_tactics'])} identified")
    for tactic in risk['mitre_tactics']:
        print(f"  - {tactic}")

    print(f"\n--- Recommendations ---")
    for rec in risk['recommendations']:
        print(f"  • {rec}")

    print(f"\n--- Analysis Complete ---")
    print(f"Total processing time: {time.time() - report['analysis_timestamp']:.3f}s")
    print(f"Framework identified {len(detections)} potential threats")

if __name__ == "__main__":
    simulate_malware_analysis()